# Call Center Dataset Analysis


### Key Terminology Clarifications:
* **IVR (Interactive Voice Response):** The automated voice menu system that callers interact with before reaching an agent.
* **IVRPosition:** The specific step or stage within the IVR system where the caller was positioned (or where they hung up).
* **MainOption & SubOption:** The numerical or textual choices selected by the customer in the IVR menu layout.
* **SelectedQueue:** The specific department or routing line the caller was assigned to based on their choices.
* **EnteredQueue & IsAnswered:** Status flags indicating whether the call successfully reached the queue line and whether a human agent answered.

## Call Center Dataset Data Dictionary & Feature Documentation

This document provides a comprehensive breakdown of the variables and features captured within the call center dataset. The features are categorized into logical operational groups to assist in data cleaning, exploratory data analysis, and behavioral insights modeling.

---

### 1. Call & System Metadata
These fields record the base technical logs captured automatically by the telecom system upon call arrival.

* **`CALL_TIME`**: The exact date and timestamp when the customer's call entered the system.
* **`lang`**: The language preferred or selected by the caller at the beginning of the IVR interaction (e.g., `S` for Sinhala, `T` for Tamil, `E` for English, typical for telecom operators like Sri Lanka Telecom).
* **`IVRPosition`**: A specific system code indicating the last node, branch, or stage the customer reached within the interactive voice menu map before being routed or hanging up.

---

### 2. IVR Menu Selections
These variables capture user interactions with the automated Interactive Voice Response (IVR) keypad menus.

* **`mainOption`**: The primary menu option (numerical digit) selected by the caller on their phone keypad during the main IVR greeting.
* **`subOption`**: The specific sub-category or service line selected under the main option (e.g., `broadband`, `peotv` for IPTV services, `voice` for fixed/mobile lines, `extragb`, or `addon`).

---

### 3. Account & Service Status Flags
These attributes are retrieved dynamically from the Customer Relationship Management (CRM) and billing databases when the caller's account identifier is recognized.

* **`validAccount`**: A boolean value (`True`/`False`) indicating whether the system recognized the customer identifier or phone number as an active, legitimate account.
* **`outstanding`**: The current unpaid balance or financial dues on the customer's account. Negative values represent an overpayment or credit balance.
* **`existingFault`**: Indicates (`YES` or empty) if there is already an open, unresolved technical issue or complaint ticket logged for this customer's service.
* **`suspendedAsset`**: Indicates (`YES` or empty) whether the customer's connection or a service asset has been temporarily deactivated or suspended (often due to outstanding bills or contractual requests).
* **`dataLimitExceed`**: Indicates (`YES` or empty) if the customer has exhausted the high-speed data quota included in their primary internet bundle.
* **`extraGBAdded`**: Indicates (`YES` or empty) whether the customer has successfully purchased or activated an extra data top-up bundle.

---

### 4. Queue & Agent Routing Details
These flags monitor the customer's transition from automated menus to human-assisted queues.

* **`selectedQueue`**: The specific agent routing pool or department queue assigned to handle the call based on the user's IVR inputs (e.g., `S1`, `S41`).
* **`enteredQueue`**: Indicates (`YES` or empty) whether the caller successfully moved past the automated menu and entered a live hold queue waiting for a human agent.
* **`isAnswered`**: Indicates (`YES` or empty) if the call was successfully accepted and answered by a live customer service agent.

---

### 5. Call Duration Performance Metrics
These variables capture operational efficiency and key performance indicators (KPIs) associated with call duration, measured in seconds.

* **`queue_time`**: The time (in seconds) the customer spent waiting in the queue after choosing to speak with an agent until an agent responded.
* **`talk_time`**: The active duration (in seconds) of the conversation between the agent and the customer.
* **`total_time`**: The entire lifecycle duration of the call (in seconds). This encompasses the IVR automated menu navigation time, the `queue_time`, and the `talk_time`. 

> **Operational Formula:**
> The duration spent navigating the automated IVR menus is implicitly calculated using the following relationship:
> $$\text{IVR Menu Time} = \text{total\_time} - \text{queue\_time} - \text{talk\_time}$$

In [1]:
# Libraries
import pandas as pd


In [2]:
# Load the dataset
file_path = r"C:\Users\Mihiliya Jayasiri\Downloads\5000.csv"
df = pd.read_csv(file_path)

# Display the first few rows to understand the structure
df.head()

,CALL_TIME,lang,mainOption,subOption,IVRPosition,existingFault,suspendedAsset,dataLimitExceed,extraGBAdded,validAccount,outstanding,selectedQueue,enteredQueue,isAnswered,queue_time,talk_time,total_time
0,5/13/2026 23:59,S,14.0,NaN,S7,NaN,NaN,NaN,NaN,True,6467.85,S7,YES,YES,273,85,447
1,5/13/2026 23:59,S,4.0,extragb,S4112,NaN,NaN,NaN,NaN,True,2979.45,S41,NaN,NaN,0,0,60
2,5/13/2026 23:59,S,1.0,NaN,S1,NaN,NaN,NaN,NaN,True,-84.53,S1,YES,YES,225,172,480
3,5/13/2026 23:59,S,1.0,NaN,SLty1,NaN,NaN,NaN,NaN,False,NaN,S1,YES,NaN,0,0,129
4,5/13/2026 23:59,S,NaN,NaN,S,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,0,0,74


## Section 1: Initial Data Exploration
In this section, we examine the dataset's shape, basic properties, data types, and missing values across all columns to evaluate data quality and completeness.

In [3]:
# High-level structural overview
print(f"Dataset Shape: {df.shape}")

Dataset Shape: (5000, 17)


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CALL_TIME        5000 non-null   object 
 1   lang             4604 non-null   object 
 2   mainOption       3563 non-null   float64
 3   subOption        2400 non-null   object 
 4   IVRPosition      4604 non-null   object 
 5   existingFault    131 non-null    object 
 6   suspendedAsset   645 non-null    object 
 7   dataLimitExceed  51 non-null     object 
 8   extraGBAdded     279 non-null    object 
 9   validAccount     4994 non-null   object 
 10  outstanding      3388 non-null   float64
 11  selectedQueue    3572 non-null   object 
 12  enteredQueue     2713 non-null   object 
 13  isAnswered       787 non-null    object 
 14  queue_time       5000 non-null   int64  
 15  talk_time        5000 non-null   int64  
 16  total_time       5000 non-null   int64  
dtypes: float64(2),

In [5]:
df.describe()

,mainOption,outstanding,queue_time,talk_time,total_time
count,3563.000000,3.388000e+03,5000.000000,5000.000000,5000.000000
mean,3.636542,1.118705e+04,117.094600,19.227200,328.673800
std,3.571218,2.145526e+05,290.487126,62.461667,367.628622
min,1.000000,-1.250000e+04,0.000000,0.000000,0.000000
25%,2.000000,3.349000e+01,0.000000,0.000000,39.000000
50%,2.000000,3.486560e+03,0.000000,0.000000,157.000000
75%,4.000000,6.374560e+03,0.000000,0.000000,519.000000
max,14.000000,1.156483e+07,1655.000000,868.000000,2296.000000


## Section 2: Examining Unique Values per Feature
Understanding the distinct categories within each text or discrete feature helps pinpoint the scope of customer interactions and data patterns.

In [6]:
# Display unique values for all features to see their domain ranges
for col in df.columns:
    print(f"{col}: {df[col].dropna().unique()[:5]} (Total unique: {df[col].nunique()})")

CALL_TIME: ['5/13/2026 23:59' '5/13/2026 23:58' '5/13/2026 23:57' '5/13/2026 23:56'
 '5/13/2026 23:55'] (Total unique: 136)
lang: ['S' 'T' 'E'] (Total unique: 3)
mainOption: [14.  4.  1.  2.  3.] (Total unique: 8)
subOption: ['extragb' 'peotv' 'voice' 'broadband' 'addon'] (Total unique: 11)
IVRPosition: ['S7' 'S4112' 'S1' 'SLty1' 'S'] (Total unique: 99)
existingFault: ['YES'] (Total unique: 1)
suspendedAsset: ['YES'] (Total unique: 1)
dataLimitExceed: ['YES'] (Total unique: 1)
extraGBAdded: ['YES'] (Total unique: 1)
validAccount: [True False] (Total unique: 2)
outstanding: [6467.85 2979.45  -84.53 -435.55 8515.52] (Total unique: 2234)
selectedQueue: ['S7' 'S41' 'S1' 'S2' 'S3'] (Total unique: 26)
enteredQueue: ['YES'] (Total unique: 1)
isAnswered: ['YES'] (Total unique: 1)
queue_time: [273   0 225 291 229] (Total unique: 452)
talk_time: [ 85   0 172  43 101] (Total unique: 264)
total_time: [447  60 480 129  74] (Total unique: 1172)


In [7]:
# Deep dive into specific key categorical columns
for col in ["lang", "mainOption", "subOption"]:
    print(f"{col}: {df[col].dropna().unique()[:12]} (Total unique: {df[col].nunique()})")

lang: ['S' 'T' 'E'] (Total unique: 3)
mainOption: [14.  4.  1.  2.  3.  6. 11.  5.] (Total unique: 8)
subOption: ['extragb' 'peotv' 'voice' 'broadband' 'addon' '1' 'other' 'residential'
 'timeout' 'organizational' 'tele_rainbow'] (Total unique: 11)


### Questions???
1. What are the 8 mainOptions means in text?
2. What are the different types of IVR positions (specially ones that have i's and Lty meaning)?
3. What is the relationship between IVRPostion and selectedQueue?

## Section 3: Data Cleaning & Preprocessing

### 1. Standardizing Binary Flags
Features like `existingFault`, `suspendedAsset`, `dataLimitExceed`, `extraGBAdded`, `enteredQueue`, and `isAnswered` only explicitly store `"YES"`. For these columns, the missing values (`NaN`) inherently denote the absence of that condition, meaning they can safely be treated as `"NO"`.

In [8]:
# Fill missing categorical values with 'NO' for binary status indicators
only_yes_cols = ["existingFault", "suspendedAsset", "dataLimitExceed", "extraGBAdded", "enteredQueue", "isAnswered"]
df[only_yes_cols] = df[only_yes_cols].fillna("NO")

# Verify the changes
df.head()

,CALL_TIME,lang,mainOption,subOption,IVRPosition,existingFault,suspendedAsset,dataLimitExceed,extraGBAdded,validAccount,outstanding,selectedQueue,enteredQueue,isAnswered,queue_time,talk_time,total_time
0,5/13/2026 23:59,S,14.0,NaN,S7,NO,NO,NO,NO,True,6467.85,S7,YES,YES,273,85,447
1,5/13/2026 23:59,S,4.0,extragb,S4112,NO,NO,NO,NO,True,2979.45,S41,NO,NO,0,0,60
2,5/13/2026 23:59,S,1.0,NaN,S1,NO,NO,NO,NO,True,-84.53,S1,YES,YES,225,172,480
3,5/13/2026 23:59,S,1.0,NaN,SLty1,NO,NO,NO,NO,False,NaN,S1,YES,NO,0,0,129
4,5/13/2026 23:59,S,NaN,NaN,S,NO,NO,NO,NO,False,NaN,NaN,NO,NO,0,0,74


In [9]:
# Recheck column fill counts after mapping binary flags
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CALL_TIME        5000 non-null   object 
 1   lang             4604 non-null   object 
 2   mainOption       3563 non-null   float64
 3   subOption        2400 non-null   object 
 4   IVRPosition      4604 non-null   object 
 5   existingFault    5000 non-null   object 
 6   suspendedAsset   5000 non-null   object 
 7   dataLimitExceed  5000 non-null   object 
 8   extraGBAdded     5000 non-null   object 
 9   validAccount     4994 non-null   object 
 10  outstanding      3388 non-null   float64
 11  selectedQueue    3572 non-null   object 
 12  enteredQueue     5000 non-null   object 
 13  isAnswered       5000 non-null   object 
 14  queue_time       5000 non-null   int64  
 15  talk_time        5000 non-null   int64  
 16  total_time       5000 non-null   int64  
dtypes: float64(2),

### 2. Identifying and Removing Dropped Calls
When both `lang` (Language choice) and `IVRPosition` are missing, it indicates a customer who hung up immediately after dialing in without interacting with the system. We will verify if these are extremely brief calls and remove them from our core behavioral analysis.

In [10]:
# Filter the DataFrame to rows where 'lang' is null
lang_null_df = df[df['lang'].isnull()]

# Verify if 'IVRPosition' is also always missing when 'lang' is missing
is_ivr_always_null = lang_null_df['IVRPosition'].isnull().all()

print(f"Total rows where 'lang' is null: {len(lang_null_df)}")
print(f"Is 'IVRPosition' also null for all these rows? {is_ivr_always_null}")

Total rows where 'lang' is null: 396
Is 'IVRPosition' also null for all these rows? True


In [11]:
# Check if these early dropped calls are shorter than 60 seconds
null_condition = df['lang'].isnull() & df['IVRPosition'].isnull()
filtered_df = df[null_condition]

all_less_than_60 = (filtered_df['total_time'] < 60).all()

print(f"Total rows where both 'lang' and 'IVRPosition' are null: {len(filtered_df)}")
print(f"Are all of these calls shorter than 60 seconds? {all_less_than_60}")

print("\nSummary statistics of total_time (in seconds) for these dropped rows:")
print(filtered_df['total_time'].describe())

Total rows where both 'lang' and 'IVRPosition' are null: 396
Are all of these calls shorter than 60 seconds? True

Summary statistics of total_time (in seconds) for these dropped rows:
count    396.000000
mean       7.863636
std        4.388315
min        0.000000
25%        5.000000
50%        7.000000
75%       10.000000
max       21.000000
Name: total_time, dtype: float64


In [12]:
# Remove these early-dropped rows from the main DataFrame in-place
df.dropna(subset=['lang'], inplace=True)
print(f"Cleaned dataset shape: {df.shape}")

Cleaned dataset shape: (4604, 17)


In [13]:
# Verify dataset state after removal
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4604 entries, 0 to 4999
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CALL_TIME        4604 non-null   object 
 1   lang             4604 non-null   object 
 2   mainOption       3563 non-null   float64
 3   subOption        2400 non-null   object 
 4   IVRPosition      4604 non-null   object 
 5   existingFault    4604 non-null   object 
 6   suspendedAsset   4604 non-null   object 
 7   dataLimitExceed  4604 non-null   object 
 8   extraGBAdded     4604 non-null   object 
 9   validAccount     4598 non-null   object 
 10  outstanding      3388 non-null   float64
 11  selectedQueue    3571 non-null   object 
 12  enteredQueue     4604 non-null   object 
 13  isAnswered       4604 non-null   object 
 14  queue_time       4604 non-null   int64  
 15  talk_time        4604 non-null   int64  
 16  total_time       4604 non-null   int64  
dtypes: float64(2), int6

## Section 4: Analyzing Feature Relationships


### 1. Main Menu Options vs. Sub-Options
Let's discover the underlying operational hierarchy of the IVR system by tracking which `subOption` selections belong under specific `mainOption` inputs.

In [14]:
# Group by mainOption and extract lists of associated unique subOptions
sub_options_per_main = (
    df.groupby('mainOption')['subOption']
    .unique()
    .apply(lambda x: [val for val in x if pd.notnull(val)])
)

# Display the nested routing rules
for main_opt, sub_opts in sub_options_per_main.items():
    print(f"Main Option {main_opt}: {sub_opts if sub_opts else 'No sub-options available'}")

Main Option 1.0: No sub-options available
Main Option 2.0: ['peotv', 'voice', 'broadband', 'other']
Main Option 3.0: No sub-options available
Main Option 4.0: ['extragb', 'addon']
Main Option 5.0: ['residential', 'timeout', 'organizational', 'tele_rainbow']
Main Option 6.0: ['1']
Main Option 11.0: No sub-options available
Main Option 14.0: No sub-options available


### Questions???

1. What does each main option mean?
2. Are there  any other suboptions that are not available here?
3. What is meant by sub option 1 in main option 6?
4. What are the meanings of sub options in 5th main option?

### 1.1 Finding: Behavior of Null Main Menu Choices
When a customer does not make a choice on the main IVR menu (`mainOption` is missing), they cannot access any sub-menus, and their status flag signals stay clear.

In [15]:
# Filter rows where mainOption is missing
null_main_df = df[df['mainOption'].isnull()]
total_rows = len(null_main_df)

# Confirm subOption is also null
is_sub_always_null = null_main_df['subOption'].isnull().all()

# Check account metrics status columns
status_cols = ["existingFault", "suspendedAsset", "dataLimitExceed", "extraGBAdded"]
are_flags_no_or_null = all(
    null_main_df[col].isnull().all() or (null_main_df[col] == "NO").all() 
    for col in status_cols
)

print(f"Total rows where 'mainOption' is null: {total_rows}")
print(f"1. Is 'subOption' definitely null? {is_sub_always_null}")
print(f"2. Are all status flags ('existingFault', 'suspendedAsset', etc.) 'NO'? {are_flags_no_or_null}")

Total rows where 'mainOption' is null: 1041
1. Is 'subOption' definitely null? True
2. Are all status flags ('existingFault', 'suspendedAsset', etc.) 'NO'? True


Similarly, for any call record where a `subOption` is absent, the account status metrics (`existingFault`, `suspendedAsset`, `dataLimitExceed`, `extraGBAdded`.) uniformly default to `"NO"`.

In [16]:
# Filter rows where subOption is null
null_sub_df = df[df['subOption'].isnull()]
total_null_sub_rows = len(null_sub_df)

# Check status columns consistency
are_all_flags_no_or_null = all(
    null_sub_df[col].isnull().all() or (null_sub_df[col] == "NO").all()
    for col in status_cols
)

print(f"Total rows where 'subOption' is null: {total_null_sub_rows}")
print(f"Is it TRUE that all account flags are empty/NO when subOption is null? {are_all_flags_no_or_null}")

print("\nDetailed column breakdown:")
for col in status_cols:
    non_null_count = null_sub_df[col].notnull().sum()
    print(f" - Column '{col}' contains {non_null_count} active entries.")

Total rows where 'subOption' is null: 2204
Is it TRUE that all account flags are empty/NO when subOption is null? True

Detailed column breakdown:
 - Column 'existingFault' contains 2204 active entries.
 - Column 'suspendedAsset' contains 2204 active entries.
 - Column 'dataLimitExceed' contains 2204 active entries.
 - Column 'extraGBAdded' contains 2204 active entries.


Also account metrics `existingFault`, `suspendedAsset` and `dataLimitExceed` have no relationship, if one have `"YES"` other don't. As of `dataLimitExceed` as `"YES"`,it has records with `extraGBAdded` as `"YES"`.

### 2. Deep Dive: IVR Position Menu Nodes
Let's see what unique system steps (`IVRPosition`) are triggered across the different core menu paths chosen by callers.

In [17]:
# Group by mainOption (retaining missing choices as a distinct group) to log active IVR positions
ivr_per_main = (
    df.groupby('mainOption', dropna=False)['IVRPosition']
    .unique()
    .apply(lambda x: [val for val in x if pd.notnull(val)])
)

print("--- IVR Positions mapped per Main Menu Choice ---")
for main_opt, ivr_positions in ivr_per_main.items():
    label = "Null / Missing Option" if pd.isnull(main_opt) else f"Main Option {main_opt}"
    print(f"\n{label} (Total unique positions: {len(ivr_positions)}):")
    print(sorted(list(ivr_positions)))

--- IVR Positions mapped per Main Menu Choice ---

Main Option 1.0 (Total unique positions: 9):
['ELty1', 'MOBITEL_ELty1', 'MOBITEL_S1', 'MOBITEL_SLty1', 'MOBITEL_T1', 'MOBITEL_TLty1', 'S1', 'SLty1', 'TLty1']

Main Option 2.0 (Total unique positions: 34):
['E21', 'E22', 'E221', 'E222', 'E23', 'E24', 'E2t', 'MOBITEL_E21', 'MOBITEL_S21', 'MOBITEL_S211', 'MOBITEL_S22', 'MOBITEL_S221', 'MOBITEL_S23', 'MOBITEL_S24', 'MOBITEL_S2t', 'MOBITEL_T2', 'S2', 'S21', 'S211', 'S22', 'S221', 'S222', 'S22i', 'S23', 'S24', 'S2i', 'S2t', 'T21', 'T211', 'T22', 'T221', 'T222', 'T23', 'T231']

Main Option 3.0 (Total unique positions: 5):
['E3', 'MOBITEL_E3', 'MOBITEL_S3', 'S3', 'T3']

Main Option 4.0 (Total unique positions: 34):
['E411', 'E4111', 'E4112', 'E41121', 'E412', 'E41211', 'E41t', 'MOBITEL_E4112', 'MOBITEL_S41', 'MOBITEL_S4111', 'MOBITEL_S4112', 'MOBITEL_S41122', 'MOBITEL_S41t', 'S41', 'S411', 'S4111', 'S411121', 'S4112', 'S41121', 'S41122', 'S411223', 'S4113', 'S41131', 'S412', 'S41211', 'S41213'

In [18]:
# Group by subOption to trace position mappings
ivr_per_sub = (
    df.groupby('subOption', dropna=False)['IVRPosition']
    .unique()
    .apply(lambda x: [val for val in x if pd.notnull(val)])
)

print("--- IVR Positions mapped per Sub-Option Category ---")
for sub_opt, ivr_positions in ivr_per_sub.items():
    label = "Null / Missing Sub-Option" if pd.isnull(sub_opt) else f"Sub-Option '{sub_opt}'"
    print(f"\n{label} (Total unique positions: {len(ivr_positions)}):")
    print(sorted(list(ivr_positions)))

--- IVR Positions mapped per Sub-Option Category ---

Sub-Option '1' (Total unique positions: 1):
['S1']

Sub-Option 'addon' (Total unique positions: 6):
['E412', 'E41211', 'S412', 'S41211', 'S41213', 'S4122']

Sub-Option 'broadband' (Total unique positions: 12):
['E22', 'E221', 'E222', 'MOBITEL_S22', 'MOBITEL_S221', 'S22', 'S221', 'S222', 'S22i', 'T22', 'T221', 'T222']

Sub-Option 'extragb' (Total unique positions: 26):
['E411', 'E4111', 'E4112', 'E41121', 'E41t', 'MOBITEL_E4112', 'MOBITEL_S4111', 'MOBITEL_S4112', 'MOBITEL_S41122', 'MOBITEL_S41t', 'S411', 'S4111', 'S411121', 'S4112', 'S41121', 'S41122', 'S411223', 'S4113', 'S41131', 'S41i', 'S41t', 'T411', 'T4111', 'T4112', 'T4113', 'T41t']

Sub-Option 'organizational' (Total unique positions: 1):
['S51']

Sub-Option 'other' (Total unique positions: 7):
['E24', 'E2t', 'MOBITEL_S24', 'MOBITEL_S2t', 'S24', 'S2i', 'S2t']

Sub-Option 'peotv' (Total unique positions: 8):
['E21', 'MOBITEL_E21', 'MOBITEL_S21', 'MOBITEL_S211', 'S21', 'S211', 

### 2.2 Finding: Calls Hanging Up on Root Language Screen
When **both** choices are completely missing, the customer only reaches the base language routing menu nodes (`'S'`, `'T'`, `'E'`, or their mobile counterparts).

In [19]:
# Filter rows where both options are null
null_both_df = df[df['mainOption'].isnull() & df['subOption'].isnull()]
unique_positions = null_both_df['IVRPosition'].unique()

print(f"Total rows where both options are null: {len(null_both_df)}")
print(f"\nUnique IVRPosition values reached:\n{unique_positions}")

print("\nFrequency breakdown of these root positions:")
print(null_both_df['IVRPosition'].value_counts(dropna=False))

Total rows where both options are null: 1041

Unique IVRPosition values reached:
['S' 'T' 'MOBITEL_S' 'E' 'MOBITEL_E' 'MOBITEL_T']

Frequency breakdown of these root positions:
IVRPosition
S            920
T             54
MOBITEL_S     31
E             29
MOBITEL_E      6
MOBITEL_T      1
Name: count, dtype: int64


### 3. Valid Account Mappings vs. Outstanding Balances & Queues
Here, we check how account validity relates to financial balances and routing rules.

In [20]:
# Filter rows where validAccount is False
false_account_df = df[df['validAccount'].astype(str).str.upper() == 'FALSE']

# Check if outstanding balance is always missing
is_outstanding_always_null = false_account_df['outstanding'].isnull().all()

print(f"Total rows where 'validAccount' is False: {len(false_account_df)}")
print(f"Is 'outstanding' balance always null for these rows? {is_outstanding_always_null}")

Total rows where 'validAccount' is False: 1210
Is 'outstanding' balance always null for these rows? True


### 3.1 Finding: Behavior of Null Account Validity
When `validAccount` is missing entirely, it is highly isolated: it only occurs under `mainOption` 6.0 and specifically routes callers directly to an emergency/specialized queue named `"ECS"`.

In [21]:
# Filter rows where validAccount is null
null_account_df = df[df['validAccount'].isnull()]
total_null_account_rows = len(null_account_df)

is_always_main_6 = (null_account_df['mainOption'] == 6.0).all()
is_always_ecs = (null_account_df['selectedQueue'] == "ECS").all()

print(f"Total rows where 'validAccount' is null: {total_null_account_rows}")
print(f"1. Do they all belong to mainOption 6? {is_always_main_6}")
print(f"2. Is the selectedQueue always 'ECS'? {is_always_ecs}")

Total rows where 'validAccount' is null: 6
1. Do they all belong to mainOption 6? True
2. Is the selectedQueue always 'ECS'? True


### 3.2 Finding: Restrictions for False Accounts
When `validAccount` is explicitly flagged as `False`, callers are strictly restricted. They only have access to sub-options `'other'` and `'extragb'`, and all flags are `"NO"`.

In [22]:
# Gather unique sub-options for False accounts
actual_sub_options = [opt for opt in false_account_df['subOption'].unique() if pd.notnull(opt)]
expected_sub_options = ['other', 'extragb']
is_sub_option_matching = set(actual_sub_options) == set(expected_sub_options)

print(f"1. Are the only active sub-options 'other' and 'extragb'? {is_sub_option_matching} (Found: {actual_sub_options})")

1. Are the only active sub-options 'other' and 'extragb'? True (Found: ['extragb', 'other'])


### Questions???

1. What is ECS? Why does it only come under main option 6?
2. What is included in other suboption?

### 4. Exploring SelectedQueue Routing Rules
Finally, we analyze rows with missing `selectedQueue` data to verify if these indicate calls where the user abandoned the system at the entry menu phase.

In [23]:
# Filter rows where selectedQueue is missing
null_queue_df = df[df['selectedQueue'].isnull()]
total_null_queue_rows = len(null_queue_df)

print(f"Total rows where 'selectedQueue' is null: {total_null_queue_rows}")

Total rows where 'selectedQueue' is null: 1033


If a call leaves the system before being assigned to a `selectedQueue`, it never queues up or gets answered, resulting in exactly `0` wait or talk times.

In [24]:
# Check handling and queue metrics
is_entered_queue_no = (null_queue_df['enteredQueue'] == "NO").all()
is_answered_no = (null_queue_df['isAnswered'] == "NO").all()
is_queue_time_zero = (null_queue_df['queue_time'] == 0).all()
is_talk_time_zero = (null_queue_df['talk_time'] == 0).all()

print(f"1. Are 'enteredQueue' values all 'NO'? {is_entered_queue_no}")
print(f"2. Are 'isAnswered' values all 'NO'? {is_answered_no}")
print(f"3. Are all 'queue_time' values exactly 0? {is_queue_time_zero}")
print(f"4. Are all 'talk_time' values exactly 0? {is_talk_time_zero}")

1. Are 'enteredQueue' values all 'NO'? True
2. Are 'isAnswered' values all 'NO'? True
3. Are all 'queue_time' values exactly 0? True
4. Are all 'talk_time' values exactly 0? True


In [27]:
# Confirm their position tree values match expected root nodes
unique_positions_null_queue = null_queue_df['IVRPosition'].unique()
expected_list = ['E', 'MOBITEL_E', 'MOBITEL_S', 'MOBITEL_T', 'S', 'T']
is_exactly_matching = set(unique_positions_null_queue) == set(expected_list)

print(f"Do the position values match the root language menus? {is_exactly_matching}")
print(f"IVRpositions found when selectedQueue is null: {unique_positions_null_queue}")

Do the position values match the root language menus? True
IVRpositions found when selectedQueue is null: ['S' 'T' 'MOBITEL_S' 'E' 'MOBITEL_E' 'MOBITEL_T']


### 5. Final Mappings Validation
A final calculation check shows a perfect breakdown split between outstanding accounts.

In [26]:
# Count condition where validAccount is False OR Null
condition = (df['validAccount'].astype(str).str.upper() == 'FALSE') | df['validAccount'].isnull()
subset_df = df[condition]

print(f"Total rows where 'validAccount' is False or Null: {len(subset_df)}")
print(f"Number of null values in 'outstanding' for these rows: {subset_df['outstanding'].isnull().sum()}")

Total rows where 'validAccount' is False or Null: 1216
Number of null values in 'outstanding' for these rows: 1216


### Questions???

1. Is this dataset is of 1212 dials?
2. Is the reason for validAccounts to be false and null because those calls were from non sltm numbers?

## New Column Recommendations


1. IVR Menu Time
2. Separate date and time